In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Intel only
# !pip install scikit-learn-intelex
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


# Training on 999 features
This is the code that get us 0.5678 on public leaderboard

In [3]:
import sys 
import os
sys.path.append('..')
import gc

In [4]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import time
import re

### Some setting for the pipeline

**Warning**: Set `USING_CACHE` = True only for resample and parameter tuning. Use with causion

In [5]:
UPSAMPLE_LATER = True 
UPSAMPLE_RATIO = 1

RETRAIN_KNN = True
RETRAIN_TRANSFORM = True

USING_SMOTE = False
FILL_MEAN = True

USING_CACHE = False # Only change to True when you are sure that the cache is up-to-date

In [6]:
from utils.check_feature import power_scaler_col

## Read the file

File has bigger feature usually has higher version. Check out carefully

Eg: For Bureau, the bureau v3.1 has less feature than the v4

For small version, we dont use `app_prev_app` dataframe, so uncomment it to add to the pipeline

In [7]:
appl_train = pd.read_csv('../data/dseb63_application_train.csv', index_col=0)

# ====== BUREAU ======

bureau = pd.read_parquet('../data/dseb63_bureau_general_v3_1.parquet')
# bureau = pd.read_parquet('../../data/dseb63_bureau_general_v4.parquet')
bureau_columns = bureau.columns

new_bureau_columns = {col: 'BUREAU_' + col for col in bureau_columns if col != 'SK_ID_CURR'} 
bureau.rename(columns=new_bureau_columns, inplace=True)

# ====== PREVIOUS APPLICATION ======

prev_app = pd.read_parquet('../data/prev_app_hanh_v1_3.parquet')
# prev_app = pd.read_parquet('../../data/prev_app_hanh_v2.parquet')
prev_app_columns = prev_app.columns

new_prev_app_columns = {col: 'PREV_APP_' + col for col in prev_app_columns if col != 'SK_ID_CURR'}
prev_app.rename(columns=new_prev_app_columns, inplace=True)

# ====== INSTALLMENTS PAYMENTS ======

installments = pd.read_parquet('../data/dseb63_installment_gb_v1_1.parquet')
# installments = pd.read_csv('../../data/dseb63_installment_gb_v2.csv')
installments_columns = installments.columns

installments.drop(columns= [col for col in installments_columns if 'TARGET' in col], inplace=True)

new_installments_columns = {col: 'INSTALLMENTS_' + col for col in installments_columns if col != 'SK_ID_CURR'}
installments.rename(columns=new_installments_columns, inplace=True)

# ====== CREDIT CARD BALANCE ======

credit_card = pd.read_parquet('../data/dseb63_credit_card_balance_gb_v1_3.parquet')
# credit_card = pd.read_parquet('../../data/dseb63_credit_card_balance_gb_v2.parquet')
credit_card_columns = credit_card.columns

new_credit_card_columns = {col: 'CREDIT_CARD_' + col for col in credit_card_columns if col != 'SK_ID_CURR'}
credit_card.rename(columns=new_credit_card_columns, inplace=True)

# ====== POS CASH BALANCE ======

pos_cash = pd.read_parquet('../data/dseb63_pos_cash_gb_v2_1.parquet') # Good
pos_cash_columns = pos_cash.columns

new_posh_cash_columns = {col: 'POS_CASH_' + col for col in pos_cash_columns if col != 'SK_ID_CURR'}
pos_cash.rename(columns=new_posh_cash_columns, inplace=True)

# ====== MERGE APP PREV APP ======

app_prev_app = pd.read_parquet('../data/dseb63_app_prev_app_features.parquet')
app_prev_app_columns = app_prev_app.columns

new_app_prev_app_columns = {col: 'APP_PREV_APP_' + col for col in app_prev_app_columns if col != 'SK_ID_CURR'}
app_prev_app.rename(columns=new_app_prev_app_columns, inplace=True)

# ====== MERGE DATA ======

df = appl_train.merge(bureau, on='SK_ID_CURR', how='left')
df = df.merge(installments, on='SK_ID_CURR', how='left')
df = df.merge(pos_cash, on='SK_ID_CURR', how='left')
df = df.merge(credit_card, on='SK_ID_CURR', how='left')
df = df.merge(prev_app, on='SK_ID_CURR', how='left')


# ======== MERGE APP PREV APP ========
# df = df.merge(app_prev_app, on='SK_ID_CURR', how='left')


In [8]:
def RELU(series):
    return series.apply(lambda x: max(0, x))
    

## Create Feature based on APPLICATION TRAIN 

In [ ]:
# from tqdm import tqdm 
# #Add agg recipes outside 
# AGGREGATION_RECIPIES = [
#     (['CODE_GENDER', 'NAME_EDUCATION_TYPE'], [('AMT_ANNUITY', 'max'),
#                                               ('AMT_CREDIT', 'max'),
#                                               ('EXT_SOURCE_1', 'mean'),
#                                               ('EXT_SOURCE_2', 'mean'),
#                                               ('OWN_CAR_AGE', 'max'),
#                                               ('OWN_CAR_AGE', 'sum')]),
#     (['CODE_GENDER', 'ORGANIZATION_TYPE'], [('AMT_ANNUITY', 'mean'),
#                                             ('AMT_INCOME_TOTAL', 'mean'),
#                                             ('DAYS_REGISTRATION', 'mean'),
#                                             ('EXT_SOURCE_1', 'mean')]),
#     (['CODE_GENDER', 'REG_CITY_NOT_WORK_CITY'], [('AMT_ANNUITY', 'mean'),
#                                                  ('CNT_CHILDREN', 'mean'),
#                                                  ('DAYS_ID_PUBLISH', 'mean')]),
#     (['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE', 'REG_CITY_NOT_WORK_CITY'], [('EXT_SOURCE_1', 'mean'),
#                                                                                            ('EXT_SOURCE_2', 'mean')]),
#     (['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE'], [('AMT_CREDIT', 'mean'),
#                                                   ('AMT_REQ_CREDIT_BUREAU_YEAR', 'mean'),
#                                                   ('APARTMENTS_AVG', 'mean'),
#                                                   ('BASEMENTAREA_AVG', 'mean'),
#                                                   ('EXT_SOURCE_1', 'mean'),
#                                                   ('EXT_SOURCE_2', 'mean'),
#                                                   ('EXT_SOURCE_3', 'mean'),
#                                                   ('NONLIVINGAREA_AVG', 'mean'),
#                                                   ('OWN_CAR_AGE', 'mean'),
#                                                   ('YEARS_BUILD_AVG', 'mean')]),
#     (['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE', 'REG_CITY_NOT_WORK_CITY'], [('ELEVATORS_AVG', 'mean'),
#                                                                             ('EXT_SOURCE_1', 'mean')]),
#     (['OCCUPATION_TYPE'], [('AMT_ANNUITY', 'mean'),
#                            ('CNT_CHILDREN', 'mean'),
#                            ('CNT_FAM_MEMBERS', 'mean'),
#                            ('DAYS_BIRTH', 'mean'),
#                            ('DAYS_EMPLOYED', 'mean'),
#                            ('DAYS_ID_PUBLISH', 'mean'),
#                            ('DAYS_REGISTRATION', 'mean'),
#                            ('EXT_SOURCE_1', 'mean'),
#                            ('EXT_SOURCE_2', 'mean'),
#                            ('EXT_SOURCE_3', 'mean')]),
# ]

In [ ]:
    # groupby_aggregate_names = []
    # for groupby_cols, specs in tqdm(AGGREGATION_RECIPIES):
    #     group_object = df.groupby(groupby_cols)
    #     for select, agg in tqdm(specs):
    #         groupby_aggregate_name = '{}_{}_{}'.format('_'.join(groupby_cols), agg, select)
    #         df = df.merge(group_object[select]
    #                             .agg(agg)
    #                             .reset_index()
    #                             .rename(index=str,
    #                                     columns={select: groupby_aggregate_name})
    #                             [groupby_cols + [groupby_aggregate_name]],
    #                             on=groupby_cols,
    #                             how='left')
    #         groupby_aggregate_names.append(groupby_aggregate_name)

    # diff_feature_names = []
    # for groupby_cols, specs in tqdm(AGGREGATION_RECIPIES):
    #     for select, agg in tqdm(specs):
    #         if agg in ['mean','median','max','min']:
    #             groupby_aggregate_name = '{}_{}_{}'.format('_'.join(groupby_cols), agg, select)
    #             diff_name = '{}_diff'.format(groupby_aggregate_name)
    #             abs_diff_name = '{}_abs_diff'.format(groupby_aggregate_name)

    #             df[diff_name] = df[select] - df[groupby_aggregate_name] 
    #             df[abs_diff_name] = np.abs(df[select] - df[groupby_aggregate_name]) 

    #             diff_feature_names.append(diff_name)
    #             diff_feature_names.append(abs_diff_name)

In [9]:
def process_df(df):
    # df = df.copy()
    
    df['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
    # df['FONDKAPREMONT_MODE'].fillna('Unknown', inplace=True)
    # df['WALLSMATERIAL_MODE'].fillna('Not Specified', inplace=True)
    df['OCCUPATION_TYPE'].replace('IT staff', 'High skill tech staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('Realty agents', 'Sales staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('HR staff', 'Laborers', inplace=True)
    # df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)
    df['OCCUPATION_TYPE'].replace(['Cleaning staff', 'Cooking staff', 'Waiters/barmen staff'], 'F&B staff', inplace=True)
    

    df['ORGANIZATION_TYPE'].replace('XNA', 'Unknown', inplace=True)
    # df['HOUSETYPE_MODE'].replace(['block of flats', 'specific housing'], 'house', inplace=True)
    
    df['WALLSMATERIAL_MODE'].replace(['Others', 'Mixed', 'Monolithic'], 'Others', inplace=True)
    df['WALLSMATERIAL_MODE'].replace(['Block', 'Stone, brick'], 'Brick', inplace=True)

    map_week_day = {
    'MONDAY': 'week_day',
    'TUESDAY': 'week_day',
    'WEDNESDAY': 'week_day',
    'THURSDAY': 'week_day',
    'FRIDAY': 'week_day',
    'SATURDAY': 'weekend',
    'SUNDAY': 'weekend',
    }
    
    

    df['WEEKDAY_APPR_PROCESS_START'] = df['WEEKDAY_APPR_PROCESS_START'].map(map_week_day)

    map_edu = {
        'Lower secondary': 0,
        'Secondary / secondary special': 1,
        'Incomplete higher': 2,
        'Higher education': 3,
        'Academic degree': 5
    }

    df['NAME_EDUCATION_TYPE'] = df['NAME_EDUCATION_TYPE'].map(map_edu, na_action='ignore').astype(int).fillna(0)

    df['NAME_FAMILY_STATUS'].replace('Unknown', 'Single / not married', inplace=True)
    df['CODE_GENDER'].replace('XNA', 'F', inplace=True)
    
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(365243)
    # df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(0, 365243)

    others = df['NAME_INCOME_TYPE'].value_counts().index[4:]
    df['NAME_INCOME_TYPE'].replace(others, 'Others', inplace=True)
    df['NAME_TYPE_SUITE'].fillna('Unaccompanied', inplace=True)

    df['OWN_CAR_AGE'].fillna(-100, inplace=True)
    df['TOTALAREA_MODE'].fillna(0, inplace=True)
    
    df['AGE_INT'] = -df['DAYS_BIRTH'] // 365
    # Steal code
    df['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan, inplace=True)
    
    def group_organizations(org_type):
        if 'Trade' in org_type:
            return 'Trade'
        elif 'Industry' in org_type:
            return 'Industry'
        elif 'Business' in org_type:
            return 'Business Entity'
        elif 'Transport' in org_type:
            return 'Transport'
        elif 'University' in org_type:
            return 'School'
        elif org_type in ['Police', 'Electricity', 'Culture', 'Religion', 'Telecom', 'Emergency', 'Mobile', 'Postal']:
            return 'Public Sector'

        else:
            return org_type
    
    df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].apply(group_organizations)
    df['NAME_TYPE_SUITE'] = df['NAME_TYPE_SUITE'].replace(['Other_A', 'Other_B'], 'Other')
    
    med_income = df.groupby(['ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE'])['AMT_INCOME_TOTAL'].transform('median')
    med_income2 = df.groupby('ORGANIZATION_TYPE')['AMT_INCOME_TOTAL'].transform('median')
    df['income_ratio'] = df['AMT_INCOME_TOTAL'] / med_income
    df['income_ratio2'] = df['AMT_INCOME_TOTAL'] / med_income2
    df['true_annuity_div_income'] = df['AMT_ANNUITY'] / med_income
    df['true_annuity_div_income2'] = df['AMT_ANNUITY'] / med_income2
    df['true_income_div_totalarea'] = med_income / df['TOTALAREA_MODE'].clip(0.001,1)
    df['true_income_div_totalarea2'] = med_income2 / df['TOTALAREA_MODE'].clip(0.001,1)
    
    df['annuity_income_percentage'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['car_to_birth_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_BIRTH'])
    df['car_to_employ_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_EMPLOYED'])
    df['children_ratio'] = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS']
    df['credit_to_annuity_ratio'] = df['AMT_CREDIT'] / df['AMT_ANNUITY'] # check
    df['credit_to_goods_ratio'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['credit_to_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    
    
    df['ext_sources_mean'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    df['ext_sources_sum'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].sum(axis=1)
    df['ext_sources_var'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].var(axis=1)
    df['ext_sources_weighted'] = df.EXT_SOURCE_1 * 2 + df.EXT_SOURCE_2 * 3 + df.EXT_SOURCE_3 * 4
    df['EXT_SOURCE_MISSING_VALUES'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isna().sum(axis=1)
    
    df['income_credit_percentage'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    df['income_per_child'] = df['AMT_INCOME_TOTAL'] / (df['CNT_CHILDREN'] + 1e-5) * (df['CNT_CHILDREN'] > 0) 
    df['income_per_person'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['phone_to_birth_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH'])
    df['phone_to_employ_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_EMPLOYED'] + 1e-5)) * (df['DAYS_EMPLOYED'] != 0) 
    
    df['cnt_non_child'] = df['CNT_FAM_MEMBERS'] - df['CNT_CHILDREN']
    df['child_to_non_child_ratio'] = df['CNT_CHILDREN'] / df['cnt_non_child'] * (df['cnt_non_child'] > 0)
    df['income_per_non_child'] = df['AMT_INCOME_TOTAL'] / df['cnt_non_child']* (df['cnt_non_child'] > 0)
    df['credit_per_person'] = df['AMT_CREDIT'] / df['CNT_FAM_MEMBERS']* (df['cnt_non_child'] > 0)
    df['credit_per_child'] = df['AMT_CREDIT'] / (1 + df['CNT_CHILDREN'])
    df['credit_per_non_child'] = df['AMT_CREDIT'] / df['cnt_non_child']* (df['cnt_non_child'] > 0)

    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['DEBT_BURDEN_PER_WORKING_DAY'] = df['PAYMENT_RATE'] / (df['DAYS_EMPLOYED'] + 1e-5) * (df['DAYS_EMPLOYED'] != 0) 
    df['DEBT_BURDEN_PER_LIFE_DAY'] = df['PAYMENT_RATE'] / df['DAYS_BIRTH']
    df['CREDIT_GOODS_PRICE_RATIO1'] = (df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']) /  df['AMT_GOODS_PRICE']
    df['CREDIT_GOODS_PRICE_RATIO2'] = (df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']) /  df['AMT_CREDIT']
    df['CREDIT_DOWN_PAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']

    df['sin_HOUR_APPR_PROCESS_START'] = np.sin(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df['cos_HOUR_APPR_PROCESS_START'] = np.cos(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df.drop(columns=['HOUR_APPR_PROCESS_START'], inplace=True)

    docs = [f for f in df.columns if 'FLAG_DOC' in f]
    df['NEW_DOC_IND_AVG'] = df[docs].mean(axis=1)
    df['NEW_DOC_IND_STD'] = df[docs].std(axis=1)
    df['NEW_DOC_IND_KURT'] = df[docs].kurtosis(axis=1)
    df['HAS_DOCUMENT'] = df[docs].max(axis=1)
    df['DOCUMENT_COUNT'] = df[docs].sum(axis=1)

    #Drop flag document  
    flag_document = [f for f in df.columns if 'FLAG_DOCUMENT_' in f]
    df.drop(columns=flag_document, inplace=True)
    
    df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'].fillna(0)
    df['LANDAREA_AVG'] = df['LANDAREA_AVG'].fillna(0)
    df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'].fillna(0)
    df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'].fillna(0)
    df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'].fillna(0)

    df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAREA_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAREA_AVG'] / df['LIVINGAREA_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_LANDAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LANDAREA_AVG'].clip(0.05,1) +1e-5) * (df['LANDAREA_AVG'] / df['LANDAREA_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_FLOORSMAX_AVG_AVG'] = (df['AMT_GOODS_PRICE'] / df['FLOORSMAX_AVG'].clip(0.05,1) +1e-5) * (df['FLOORSMAX_AVG'] / df['FLOORSMAX_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAPARTMENTS_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAPARTMENTS_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAPARTMENTS_AVG'] / df['LIVINGAPARTMENTS_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG'] = (df['AMT_GOODS_PRICE'] / df['YEARS_BUILD_AVG'].clip(0.05,1) +1e-5) * (df['YEARS_BUILD_AVG'] / df['YEARS_BUILD_AVG']+1e-5)
    
    # df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAREA_AVG'] = df['AMT_GOODS_PRICE'] / df['LIVINGAREA_AVG'].clip(0.001,1)
    # df['RATIO_AMT_GOODS_PRICE_TO_LANDAREA_AVG'] = df['AMT_GOODS_PRICE'] / df['LANDAREA_AVG'].clip(0.001,1)
    # df['RATIO_AMT_GOODS_PRICE_TO_FLOORSMAX_AVG_AVG'] = df['AMT_GOODS_PRICE'] / df['FLOORSMAX_AVG'].clip(0.001,1)
    # df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAPARTMENTS_AVG'] = df['AMT_GOODS_PRICE'] / df['LIVINGAPARTMENTS_AVG'].clip(0.001,1) 
    # df['RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG'] = df['AMT_GOODS_PRICE'] / df['YEARS_BUILD_AVG'].clip(0.001,1)
    
    df['RELIABILITY_IN_CUSTOMER_CITY'] = df['REG_CITY_NOT_LIVE_CITY'] + df['REG_CITY_NOT_WORK_CITY'] + df['REG_REGION_NOT_LIVE_REGION'] + df['REG_REGION_NOT_WORK_REGION'] + df['LIVE_CITY_NOT_WORK_CITY'] + df['LIVE_REGION_NOT_WORK_REGION']
    df['SUM_CONTACTS'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE'] + df['FLAG_CONT_MOBILE'] + df['FLAG_PHONE'] + df['FLAG_EMAIL']

    #Some features from bureau 
    df['TOTAL_ENQUIRIES_CREDIT_BUREAU'] = df[['AMT_REQ_CREDIT_BUREAU_DAY',
                                            'AMT_REQ_CREDIT_BUREAU_HOUR',
                                            'AMT_REQ_CREDIT_BUREAU_WEEK',
                                            'AMT_REQ_CREDIT_BUREAU_MON',
                                            'AMT_REQ_CREDIT_BUREAU_QRT',
                                            'AMT_REQ_CREDIT_BUREAU_YEAR']].sum(axis=1)
    
    
        
    # df['PCTG_ENQUIRIES_HOUR'] = df['AMT_REQ_CREDIT_BUREAU_HOUR'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    # df['PCTG_ENQUIRIES_DAY'] = df['AMT_REQ_CREDIT_BUREAU_DAY'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_WEEK'] = df['AMT_REQ_CREDIT_BUREAU_WEEK'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_MON'] = df['AMT_REQ_CREDIT_BUREAU_MON'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_QRT'] = df['AMT_REQ_CREDIT_BUREAU_QRT'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_YEAR'] = df['AMT_REQ_CREDIT_BUREAU_YEAR'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']

    df.drop(columns=['AMT_REQ_CREDIT_BUREAU_DAY','AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_WEEK'], inplace=True)

    missing_columns = ['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
                   'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
                   'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
                   'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG']

    df['MISSING_GRADINGS'] = df[missing_columns].isna().sum(axis=1)
    # Reliability in customer city or region of residence
    df['RELIABILITY_IN_CUSTOMER_CITY'] = df['REG_CITY_NOT_LIVE_CITY'] + df['REG_CITY_NOT_WORK_CITY'] + df['REG_REGION_NOT_LIVE_REGION'] + df['REG_REGION_NOT_WORK_REGION'] + df['LIVE_CITY_NOT_WORK_CITY'] + df['LIVE_REGION_NOT_WORK_REGION']
    df['SUM_CONTACTS'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE'] + df['FLAG_CONT_MOBILE'] + df['FLAG_PHONE'] + df['FLAG_EMAIL']

    # numerical transformation
    df['DAYS_EMPLOYED'].replace(365243,0, inplace=True)
    df['days_employed_percentage'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    
    
    # df['REGION_POPULATION_RELATIVE'] = np.sqrt(df['REGION_POPULATION_RELATIVE'])
    # df['APARTMENTS_AVG'] = np.log1p(50 * df['APARTMENTS_AVG'])
    # df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'] ** 3
    # df['COMMONAREA_AVG'] = df['COMMONAREA_AVG'].clip(0.0001,1) ** (-1/5)
    # df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'] ** (1/10)
    # df['ENTRANCES_AVG'] = df['ENTRANCES_AVG'] ** (1/3)
    # df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'] ** (1/2.5)
    # df['FLOORSMIN_AVG'] = df['FLOORSMIN_AVG'] ** (1/2.2)
    # df['LANDAREA_AVG'] = df['LANDAREA_AVG'] ** (1/5)
    # df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'] ** (1/3)
    # df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'] ** (1/3)
    # df['NONLIVINGAPARTMENTS_AVG'] = df['NONLIVINGAPARTMENTS_AVG'] ** (1/5)
    # df['NONLIVINGAREA_AVG'] = df['NONLIVINGAREA_AVG'] ** (1/3)
    # df['OBS_30_CNT_SOCIAL_CIRCLE'] = df['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['DEF_30_CNT_SOCIAL_CIRCLE'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['OBS_60_CNT_SOCIAL_CIRCLE'] = df['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['DEF_60_CNT_SOCIAL_CIRCLE'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/5)

    # vuxminhan params
    df['REGION_POPULATION_RELATIVE'] = np.sqrt(df['REGION_POPULATION_RELATIVE'])
    df['APARTMENTS_AVG'] = np.log1p(50 * df['APARTMENTS_AVG'])
    # df['YEARS_BEGINEXPLUATATION_AVG'] = df['YEARS_BEGINEXPLUATATION_AVG'] ** 30 # New
    df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'] ** 3
    df['COMMONAREA_AVG'] = df['COMMONAREA_AVG'].clip(0.000001,1) ** (-1/200)
    df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'] ** (1/40)
    df['ENTRANCES_AVG'] = df['ENTRANCES_AVG'] ** (1/3)
    df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'] ** (1/2.5)
    df['FLOORSMIN_AVG'] = df['FLOORSMIN_AVG'] ** (1/2.2)
    df['LANDAREA_AVG'] = df['LANDAREA_AVG'] ** (1/5)
    df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'] ** (1/3)
    df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'] ** (1/3)
    df['NONLIVINGAPARTMENTS_AVG'] = df['NONLIVINGAPARTMENTS_AVG'] ** (1/7)
    df['NONLIVINGAREA_AVG'] = df['NONLIVINGAREA_AVG'] ** (1/5)
    df['OBS_30_CNT_SOCIAL_CIRCLE'] = df['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/7)
    df['DEF_30_CNT_SOCIAL_CIRCLE'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/7)
    df['OBS_60_CNT_SOCIAL_CIRCLE'] = df['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/7)
    df['DEF_60_CNT_SOCIAL_CIRCLE'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/7)

    
    
    return df
df = process_df(df)

In [10]:
bad_flag = ['FLAG_EMAIL', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_CONT_MOBILE', 'HOUSETYPE_MODE']
df.drop(columns=bad_flag, inplace=True)

# might merge phont emphone workphone to phone

## Apply the same pipeline to APPLICATION TEST

In [11]:
X_test_ = pd.read_csv('../data/dseb63_application_test.csv',  index_col=0)
X_test_ = X_test_.merge(bureau, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(installments, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(pos_cash, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(credit_card, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(prev_app, on='SK_ID_CURR', how='left')

# ======== MERGE APP PREV APP ========
# X_test_ = X_test_.merge(app_prev_app, on='SK_ID_CURR', how='left')
X_test_ = process_df(X_test_)

In [12]:
X_test_.drop(columns=bad_flag, inplace=True)

In [13]:
# # If you want to free memory
# del bureau, installments, pos_cash, credit_card, prev_app
# gc.collect()

## Check null value and fillna

In [14]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data

In [15]:
check_null_df = display_missing_data_info(df)
check_null_df.to_excel('../temp/check_null.xlsx')

                             Missing Values  Percentage (%)
BUREAU_BAD_DEBT_LAST_5_YEAR          245993       99.993496
BUREAU_BAD_DEBT_FINISHED             245993       99.993496
BUREAU_SOLD_DURATION                 244900       99.549203
BUREAU_SUM_DEBT_MICROLOAN            243182       98.850855
BUREAU_SUM_MICROLOAN                 243182       98.850855
...                                     ...             ...
income_per_non_child                      1        0.000406
cnt_non_child                             1        0.000406
credit_per_non_child                      1        0.000406
credit_per_person                         1        0.000406
income_per_person                         1        0.000406

[813 rows x 2 columns]


In [16]:
df.shape

(246009, 879)

### High null rate columns
Here are the columns with high null rate. We will fillna these columns will 0

In [17]:
high_null_cols = [
    'BUREAU_SOLD_SUM_CREDIT',
    'BUREAU_BAD_DEBT_LAST_5_YEAR',
    'BUREAU_BAD_DEBT_FINISHED',
    'BUREAU_BAD_DEBT_SUM_CREDIT',
    'BUREAU_CLOSE_LATENCY_30_DAYS',
    'BUREAU_SUM_OTHER',
    'BUREAU_SUM_OVERDUE_MICROLOAN',
    'BUREAU_SUM_DEBT_MICROLOAN',
    'BUREAU_SUM_MICROLOAN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MIN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MAX',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MEAN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MAX',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MIN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MEAN',
    'BUREAU_SOLD_DURATION',
    'BUREAU_COUNT_SOLD',
    'BUREAU_SOLD_LAST_1000_DAYS',
    'BUREAU_SOLD_SUM_CREDIT', # 98.3

    'BUREAU_ACTIVE_SUM_CREDIT_30_DAYS', # 97.16
    'BUREAU_ACTIVE_SUM_DEBT_30_DAYS',
    'BUREAU_ACTIVE_SUM_OVERDUE_30_DAYS',

    'BUREAU_CLOSE_LATENCY_180_DAYS', # 96.6
    'BUREAU_CLOSE_SUM_DEBT_180_DAYS', # 96.54
    'BUREAU_CLOSE_SUM_OVERDUE_180_DAYS',
    'BUREAU_CLOSE_SUM_CREDIT_180_DAYS',
    # 'BUREAU_CLOSE_LATENCY_365_DAYS', #86

    'BUREAU_SUM_MORTGAGE', # 95.35 -> 0.5644
    'BUREAU_SUM_DEBT_MORTGAGE',
    'BUREAU_SUM_OVERDUE_MORTGAGE',

    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_STD', # 94.84

    'BUREAU_SUM_CREDIT_CAR_LOAN', # 93.611
    'BUREAU_SUM_OVERDUE_CAR_LOAN',
    'BUREAU_SUM_DEBT_CAR_LOAN',

    'BUREAU_CLOSE_SUM_CREDIT_LIFE_TIME', # 91.91

    'PREV_APP_REFUSED_AMT_ANNUITY_SKEW', # 91.5
    'PREV_APP_REFUSED_AMT_GOODS_PRICE_SKEW',
    'PREV_APP_REFUSED_AMT_CREDIT_SKEW', # 89.9
    'PREV_APP_REFUSED_AMT_APPLICATION_SKEW',

    'POS_CASH_LONG_TERM_LAST_36_MONTHS', # 88.5
    'POS_CASH_LONG_TERM_CNT_INSTALMENT',
    'POS_CASH_LONG_TERM_LONG_TERM',
    'POS_CASH_LONG_TERM_SK_DPD',
    'POS_CASH_LONG_TERM_SK_DPD_DEF',
    'POS_CASH_LONG_TERM_RATE_COMPLETED',
    'POS_CASH_LONG_TERM_SK_ID_PREV',
    'POS_CASH_LONG_TERM_NUM_INSTALMENT',
    'POS_CASH_LONG_TERM_LAST_12_MONTHS', # 0.56612

    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MAX', # 85.3
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MAX',
    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MEAN',
    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MIN',
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MEAN',
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MIN',

    'PREV_APP_REFUSED_RATIO_GOODS_TO_ANNUITY_STD', # 84.7
    'PREV_APP_REFUSED_RATIO_APPLICATION_TO_ANNUITY_STD',
    'PREV_APP_REFUSED_CREDIT_ANNUITY_RATIO_STD',
    'PREV_APP_REFUSED_ANNUITY_PAYMENT_PRODUCT_STD',
    'PREV_APP_REFUSED_AMT_ANNUITY_STD',
    'PREV_APP_REFUSED_AMT_GOODS_PRICE_STD', # 0.56626

    'PREV_APP_REFUSED_APP_CREDIT_PERC_VAR', # 83.96
    'PREV_APP_REFUSED_AMT_CREDIT_STD',
    'PREV_APP_REFUSED_AMT_APPLICATION_STD',
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_sum', # 0.56654

    # 80

    # Add cc_balance_v2
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_var',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_var',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_var',
    'CREDIT_CARD_SUM_ALL_CNT_DRAWINGS_var',
    'CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_std',
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_max',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_max',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_mean',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_max',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_min',



    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_std' 
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_std',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_std',
    'CREDIT_CARD_SUM_ALL_CNT_DRAWINGS_mean',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_mean',
    'CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_mean',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_max',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_min',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_mean',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_min',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_max',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_mean',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_mean', # 0.56664

    'BUREAU_GENERAL_AMT_ANNUITY_std',
    'BUREAU_ACTIVE_SUM_OVERDUE_LIFE_TIME',
    'BUREAU_ACTIVE_SUM_CREDIT_LIFE_TIME', # 0.56670

]

In [18]:
for col in high_null_cols:
    if col in X_test_.columns:
        X_test_[col].fillna(0, inplace=True)
        df[col].fillna(0,inplace= True)

## Numerical Transformation

In [19]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, OrdinalEncoder, LabelEncoder, MinMaxScaler

In [20]:
# X = pd.read_parquet('../data/df_train.parquet')
X = df
del df

In [21]:
y = X['TARGET']
X.drop(columns=['TARGET', 'SK_ID_CURR'], inplace=True)

test_id = X_test_['SK_ID_CURR']
X_test_.drop(columns=['SK_ID_CURR'], inplace=True)


In [22]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246009 entries, 0 to 246008
Columns: 877 entries, NAME_CONTRACT_TYPE to days_employed_percentage
dtypes: float64(841), int64(22), object(14)
memory usage: 1.6+ GB


### Get columns type (int, float, object)

In [23]:
ordinal_cols = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE', 'CODE_GENDER', 'NAME_CONTRACT_TYPE']
float_cols = []
int_cols = []
cate_cols = []
flag_cols = []
for col in X.columns:
    if col not in ordinal_cols:
        if X[col].dtype == 'float64':
            float_cols.append(col)
            X[col] = X[col].astype('float64')
        elif X[col].dtype == 'int64':

            int_cols.append(col)
            X[col] = X[col].astype('int64')
        else:
            cate_cols.append(col)
            X[col] = X[col].astype('str')
    else:
        X[col] = X[col].astype('str')

In [24]:
# describe = X.describe()
# describe.columns[describe.values[-1,:]>999999998]

### STD to VAR
We test the pipeline and we found that by changing standard derivation to variance, the model perform better from 0.5675 to 0.5678

In [25]:
# Test std to var on ratio
for col in X.columns:
    if 'std' in col.lower():
        # if '_ratio' in col.lower():
        if X[col].max() < 50_000: # Avoid large number
            X[col] = X[col]**2
            X_test_[col] = X_test_[col]**2
            print(col)

BUREAU_GENERAL_CREDIT_DURATION_std
BUREAU_GENERAL_DEBT_PERCENTAGE_std
BUREAU_GENERAL_UTILIZATION_RATIO_std
INSTALLMENTS_DIFF_std
INSTALLMENTS_LATE_std
INSTALLMENTS_EARLY_std
INSTALLMENTS_LATENESS_LAST_180_DAYS_std
INSTALLMENTS_LATENESS_LAST_730_DAYS_std
INSTALLMENTS_EARLYNESS_LAST_180_DAYS_std
INSTALLMENTS_EARLYNESS_LAST_730_DAYS_std
INSTALLMENTS_PAYMENT_LATENESS_LAST_365_DAYS_std
INSTALLMENTS_PAYMENT_LATENESS_LAST_730_DAYS_std
INSTALLMENTS_PAYMENT_EARLYNESS_LAST_365_DAYS_std
INSTALLMENTS_PAYMENT_EARLYNESS_LAST_730_DAYS_std
CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_std
CREDIT_CARD_6_MONTH_AMT_INST_MIN_REGULARITY_std
CREDIT_CARD_12_MONTH_AMT_INST_MIN_REGULARITY_std
CREDIT_CARD_36_MONTH_AMT_INST_MIN_REGULARITY_std
PREV_APP_PREV_CREDIT_ANNUITY_RATIO_STD
PREV_APP_PREV_RATIO_APPLICATION_TO_ANNUITY_STD
PREV_APP_PREV_RATIO_GOODS_TO_ANNUITY_STD
PREV_APP_APPROVED_FINISH_RATE_STD
PREV_APP_APPROVED_RATIO_APPLICATION_TO_ANNUITY_STD
PREV_APP_APPROVED_RATIO_GOODS_TO_ANNUITY_STD
PREV_APP

### Find suitable column for power transformation
To not damage to much on the distribution of the data, we temporary fill the null value with mean value before testing for skewness and kurtosis

In [26]:
# TEST

X2 = X.copy()

for col in float_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
    
for col in int_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
for col in cate_cols:
    X2[col].fillna('Unknown', inplace=True)

Testing for skewness and kurtosis

In [27]:
power_col, standard_col, min_max_col = power_scaler_col(X2[float_cols+int_cols], skewness= 3, kurtosis = 20, use_cache=USING_CACHE)

100%|██████████| 863/863 [00:04<00:00, 193.25it/s]


3 / 20 Get me 566

In [28]:
len(power_col), len(standard_col), len(min_max_col)

(441, 338, 84)

In [29]:
if FILL_MEAN:
    for col in float_cols:
        X[col].fillna(X[col].mean(), inplace=True)
        X_test_[col].fillna(X[col].mean(), inplace=True)
        
    for col in int_cols:
        X[col].fillna(X[col].mean(), inplace=True)
        X_test_[col].fillna(X[col].mean(), inplace=True)
        
    for col in cate_cols:
        X[col].fillna(np.nan, inplace=True)
        X_test_[col].fillna(np.nan, inplace=True)

else:
    for col in float_cols:
        X[col].fillna(0, inplace=True)
        X_test_[col].fillna(0, inplace=True)
        
    for col in int_cols:
        X[col].fillna(0, inplace=True)
        X_test_[col].fillna(0, inplace=True)
        
    for col in cate_cols:
        X[col].fillna(np.nan, inplace=True)
        X_test_[col].fillna(np.nan, inplace=True)
    pass

In [30]:
X_columns = X.columns

### ADD KNN Feature

In [31]:
y = y.to_numpy()

In [32]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

knn_pipeline_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

In [33]:
onehot_transformer = OneHotEncoder(handle_unknown='ignore')
power_transformer = PowerTransformer()
ordinal_transformer = OrdinalEncoder()
scaler_transformer = StandardScaler()
min_max_transformer = MinMaxScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, cate_cols),
        ('power', power_transformer, power_col), # power_transformer
        ('ordinal', ordinal_transformer, ordinal_cols),
        ('scale', scaler_transformer, standard_col),
        ('min_max', min_max_transformer, min_max_col)
        
    ],
    n_jobs=-1
)




In [34]:
weak_df_col = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'credit_to_annuity_ratio', 'annuity_income_percentage']
weak_df_col_2 = ['DAYS_ID_PUBLISH', 'DAYS_REGISTRATION', 'days_employed_percentage', 'car_to_birth_ratio', 'NAME_EDUCATION_TYPE', 'AGE_INT']

## Upsample - Downsample

### Manual double
Main idea: LR is a high bias - low variance model, so we decide to employ a weak classifier to find 'obivous' and 'controversal' sample. Then we will try to resample those two type of data at different rate.

In [35]:
#from imblearn.over_sampling import SMOTENC
from sklearn.utils import resample

### Using KNN

In [36]:
def resample_data(X_train, y_train, rate = 0.2, decay = 0.7, return_idx = False, sampling_strategy = 0.1):
    
    posidx = (y_train == 1)
    negidx = (y_train == 0)

    # X_train = np.concatenate([X_train, y_train2], axis=1)
    posidx = np.where(posidx)[0]
    negidx = np.where(negidx)[0]

    n_samples = int(len(negidx) * rate)

    if posidx.sum() < n_samples: # Not enough positive samples
        return X_train, y_train
    
    if USING_SMOTE:
        if sampling_strategy == 'auto':
            sampling_strategy = 0.5 * (len(posidx) / len(negidx))

        knn = KNeighborsClassifier(n_neighbors=50, n_jobs=-1) # Train a weak KNN model
        knn.fit(X_train, y_train)
        y_temp = knn.predict_proba(X_train[posidx, :])[:,1] # Get the probability of being positive

        border_idx = np.where(y_temp < sampling_strategy)[0] # Find the border samples ()
        mask = np.zeros(len(y_temp), dtype=bool) 
        mask[border_idx] = True

        important_idx = posidx[mask]
        not_important_idx = posidx[~mask]

        len_import = len(important_idx)
        len_not_import = len(not_important_idx)
        len_data = len(posidx)

        reverse_decay = (len_data - len_not_import * decay) / len_import

        upscale_important = int(len_import/len_data * n_samples * reverse_decay)
        upscale_not_important = int(len_not_import/len_data * n_samples * decay)

        important_idx2 = resample(important_idx, n_samples=upscale_important, random_state=42)
        not_important_idx2 = resample(not_important_idx, n_samples=upscale_not_important, random_state=42)

        print('Near border sample:',len_import, 'Up to', upscale_important)
        print('Far border sample:',len_not_import, 'Up to', upscale_not_important)

        posidx2 = np.concatenate([important_idx2, not_important_idx2])
        

    else:
        posidx2 = resample(posidx, n_samples=n_samples, random_state=42)

    idx = np.concatenate([posidx2, negidx])
    np.random.shuffle(idx)
    
    if return_idx:
        return idx

    if isinstance(X_train, pd.DataFrame):
        return X_train.iloc[idx,:], y_train[idx]
    else:
        return X_train[idx,:], y_train[idx]

### Using logistic

In [37]:
from sklearn.linear_model import LogisticRegression

def log_resample_data(X_train, y_train, rate = 0.2, decay = 0.7, return_idx = False, sampling_strategy = 0.1):
    
    posidx = (y_train == 1)
    negidx = (y_train == 0)

    # X_train = np.concatenate([X_train, y_train2], axis=1)
    posidx = np.where(posidx)[0]
    negidx = np.where(negidx)[0]

    n_samples = int(len(negidx) * rate)

    if posidx.sum() < n_samples: # Not enough positive samples
        return X_train, y_train
    
    if USING_SMOTE:
        if sampling_strategy == 'auto':
            sampling_strategy = 0.5 * (len(posidx) / len(negidx))

        start = time.time()
        lr = LogisticRegression(max_iter=5000, C=0.001, tol= 1e-5, random_state=42, class_weight='balanced') # Train a weak Log model
        lr.fit(X_train, y_train)
        y_temp = lr.predict_proba(X_train[posidx, :])[:,1] # Get the probability of being positive
        end = time.time()
        print('Log fitted', end-start)


        border_idx = np.where(y_temp < sampling_strategy)[0] # Find the border samples ()
        mask = np.zeros(len(y_temp), dtype=bool) 
        mask[border_idx] = True

        important_idx = posidx[mask]
        not_important_idx = posidx[~mask]

        len_import = len(important_idx)
        len_not_import = len(not_important_idx)
        len_data = len(posidx)

        reverse_decay = (len_data - len_not_import * decay) / len_import

        upscale_important = int(len_import/len_data * n_samples * reverse_decay)
        upscale_not_important = int(len_not_import/len_data * n_samples * decay)

        important_idx2 = resample(important_idx, n_samples=upscale_important, random_state=42)
        not_important_idx2 = resample(not_important_idx, n_samples=upscale_not_important, random_state=42)

        print('Near border sample:',len_import, 'Up to', upscale_important)
        print('Far border sample:',len_not_import, 'Up to', upscale_not_important)

        posidx2 = np.concatenate([important_idx2, not_important_idx2])
        

    else:
        posidx2 = resample(posidx, n_samples=n_samples, random_state=42)

    idx = np.concatenate([posidx2, negidx])
    np.random.shuffle(idx)
    
    if return_idx:
        return idx

    if isinstance(X_train, pd.DataFrame):
        return X_train.iloc[idx,:], y_train[idx]
    else:
        return X_train[idx,:], y_train[idx]

## Feature Transformation

In [38]:
X_train = X.copy()

In [39]:
num_features = X_train.shape[1]
num_samples = X_train.shape[0]

### Adding KNN features and data transformation

#### Cache usage:
Only use cache when no change was made in the previous steps or data prep

In [40]:
if USING_CACHE \
    and os.path.exists(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy') \
    and os.path.exists(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy'):

    print('Load cached data')
    X_train = np.load(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy')
    X_test = np.load(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy')

    y = np.load(f'../temp/y_train_{num_samples}_{num_features}.npy')
    feature_names = np.load(f'../temp/feature_names_{num_samples}_{num_features}.npy', allow_pickle=True)

else:
    print('Fit weak features')
    start = time.time()
    knn_pipeline.fit(X_train[weak_df_col], y) # fit on original data
    knn_pipeline_2.fit(X_train[weak_df_col_2], y) # fit on original data


    weak_feature_train = knn_pipeline.predict_proba(X_train[weak_df_col])[:,1]
    weak_feature_val = knn_pipeline.predict_proba(X_test_[weak_df_col])[:,1]

    weak_feature_train_2 = knn_pipeline_2.predict_proba(X_train[weak_df_col_2])[:,1]
    weak_feature_val_2 = knn_pipeline_2.predict_proba(X_test_[weak_df_col_2])[:,1]

    end = time.time()
    print('Weak features fitted', end-start)

    print('Transforming data')
    start = time.time()
    
    preprocessor.fit(X) # fit on original data
    X_train = preprocessor.transform(X_train)
    X_test = preprocessor.transform(X_test_)
    end = time.time()
    print('Data transformed', end-start)


    X_train = np.concatenate([X_train, 
                                weak_feature_train.reshape(-1,1), 
                                weak_feature_train_2.reshape(-1,1)
                                ], axis=1)
    X_test = np.concatenate([X_test, 
                            weak_feature_val.reshape(-1,1), 
                            weak_feature_val_2.reshape(-1,1)
                            ], axis=1)

    cat_feature = preprocessor.named_transformers_['onehot'].get_feature_names_out(cate_cols)
    feature_names = np.concatenate([cat_feature, 
                                    power_col, 
                                    ordinal_cols, 
                                    standard_col, 
                                    min_max_col, 
                                    [
                                        'WEAK_FEATURE', 
                                        'WEAK_FEATURE_2',
                                    ]])

    np.save(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy', X_train)
    np.save(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy', X_test)

    np.save(f'../temp/y_train_{num_samples}_{num_features}.npy', y)
    np.save(f'../temp/feature_names_{num_samples}_{num_features}.npy', feature_names)


Fit weak features


Weak features fitted 23.885792016983032
Transforming data
Data transformed 74.30681419372559


In [41]:
idx = log_resample_data(X_train, y, rate = UPSAMPLE_RATIO, decay = 1.1, return_idx = True, sampling_strategy = 0.5)
X_resampled = X_train[idx,:]
y_resampled = y[idx]

In [42]:
print('Resampled data shape:', X_resampled.shape)

Resampled data shape: (452266, 945)


In [43]:
# X_resampled = pd.DataFrame(X_resampled, columns=feature_names)
# X_test = pd.DataFrame(X_test, columns=feature_names)

# X_resampled['TARGET'] = y_resampled


# y_test = pd.read_csv('../temp/target.csv', index_col=0)
# X_test['TARGET'] = y_test['True_Target'].values

# X_resampled.to_parquet('../temp/X_train.parquet', index=False)
# X_test.to_parquet('../temp/X_val.parquet', index=False)

# X_resampled.drop(columns=['TARGET'], inplace=True)
# X_test.drop(columns=['TARGET'], inplace=True)

In [44]:
# raise Exception('Stop here')

## Feature selection

Creating a bool mask for selected feature

**WARNING**: Add new features into the excel file or concatnate new feature manually

In [45]:
feature_map = np.ones(len(feature_names), dtype=bool)

In [46]:
feature_map.sum()

945

In [48]:
feature_important_df  = pd.read_excel('../temp/feature_importance.xlsx')
threshold_important = 250
threshold_split = 4

mask_important = feature_important_df['Importance'] > threshold_important
mask_split = feature_important_df['Num_Split'] > threshold_split

full_ranked_features =  feature_important_df['Feature'].values
important_features = full_ranked_features[mask_important & mask_split]
# important_features = feature_important_df[feature_important_df['Importance'] > threshold_important]['Feature'].values

Automatic add new features that not appear in *feature_importance* file (use with causion)

In [49]:
len(important_features)

850

In [50]:
feature_names_lightgbm = []
for i, feature in enumerate(feature_names):
    
    feature_names_lightgbm.append(re.sub(r'[^\w]','_', feature))
new_features = np.array([col for col in feature_names_lightgbm if col not in full_ranked_features])
important_features = np.concatenate([important_features, new_features])
feature_map = np.isin(feature_names_lightgbm, important_features)

In [51]:
for col in feature_names:
    if '_nan' in col.lower():
        feature_map[np.where(feature_names == col)[0]] = False

In [52]:
new_features

array(['OCCUPATION_TYPE_nan', 'FONDKAPREMONT_MODE_nan',
       'WALLSMATERIAL_MODE_nan',
       'BUREAU_GENERAL_CREDIT_TO_ANNUITY_RATIO_var',
       'BUREAU_GENERAL_BUREAU_CREDIT_DEBT_RATIO_var',
       'INSTALLMENTS_LATENESS_LAST_730_DAYS_max',
       'INSTALLMENTS_EARLYNESS_LAST_730_DAYS_max',
       'INSTALLMENTS_PAYMENT_LATENESS_LAST_730_DAYS_max',
       'INSTALLMENTS_PAYMENT_EARLYNESS_LAST_730_DAYS_max',
       'INSTALLMENTS_INSTALLMENT_PAYMENT_DIFF_LAST_365_DAYS_max',
       'INSTALLMENTS_INSTALLMENT_PAYMENT_DIFF_LAST_730_DAYS_max',
       'PREV_APP_PREV_DAYS_DECISION_VAR', 'BUREAU_SOLD_FINISHED'],
      dtype='<U55')

In [53]:
len(important_features)

863

In [54]:
feature_map.sum()

856

# Modelling

In [55]:
from sklearn.metrics import make_scorer, roc_auc_score
def gini_coefficient(y_true, y_pred):
    """
    Calculate the Gini coefficient using predictions and true labels.
    
    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted probabilities.
    
    Returns:
    float: Gini coefficient.
    """
    auc = roc_auc_score(y_true, y_pred)  # AUC calculation
    return 2 * auc - 1  # Gini coefficient
gini_scorer = make_scorer(gini_coefficient, needs_proba=True)

In [56]:
logistic_model = LogisticRegression(random_state=42, max_iter=10000, class_weight='balanced', C = 0.001, tol = 2e-5, n_jobs=-1)

In [57]:
logistic_model.fit(X_resampled[:, feature_map], y_resampled)
y_pred = logistic_model.predict_proba(X_test[:, feature_map])

In [58]:
submission = pd.DataFrame({
    'SK_ID_CURR': test_id,
    'TARGET': y_pred[:, 1]
})


In [61]:
ground_truth = pd.read_csv('../temp/target.csv')
submission2 = submission.merge(ground_truth, on='SK_ID_CURR', how='left')
gini_coefficient(submission2['True_Target'], submission2['TARGET'])

0.5691880425794271

In [63]:
0.5691880425794271

0.5691880425794271

In [62]:
#submission.to_csv('../submission/log_knn_oversample.csv', index=False)

In [ ]:
0.5690277886410335

0.5690277886410335

In [ ]:
# Over sampling
0.5690168839986547


0.5690168839986547